In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"]      = "3"
os.environ["GRPC_VERBOSITY"]        = "ERROR"

import warnings
warnings.filterwarnings("ignore", message=".*shared layers.*", category=UserWarning)

REPO_URL = "https://github.com/CryAndRRich/redqueen.git"

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB  = Path("/content").exists() and not ON_KAGGLE

if ON_KAGGLE:
    REPO_DIR = Path("/kaggle/working/redqueen")
elif ON_COLAB:
    REPO_DIR = Path("/content/redqueen")
else:
    REPO_DIR = Path(__file__).resolve().parent.parent \
               if "__file__" in dir() else Path.cwd()
    _here = Path.cwd()
    for _p in [_here] + list(_here.parents):
        if (_p / "agent").exists() and (_p / "src").exists():
            REPO_DIR = _p
            break

print(f"Environment : {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local'}")
print(f"REPO_DIR    : {REPO_DIR}")

if ON_KAGGLE or ON_COLAB:
    if not (REPO_DIR / ".git").exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
        print("Cloned repo.")
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
        print("Repo up to date.")

    subprocess.run([
        "pip", "install", "-q",
        "stable-baselines3>=2.2.1",
        "sb3-contrib>=2.2.1",
    ], check=True)
    print("Dependencies installed.")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

assert (REPO_DIR / "agent").exists(), f"FATAL: agent/ not found at {REPO_DIR}"
print(f"Python      : {sys.version.split()[0]}")
print("Setup complete.")


In [ ]:
from pathlib import Path

assert "REPO_DIR" in dir(), "REPO_DIR not defined — run Cell 1 first."

ARTIFACTS_DIR   = REPO_DIR / "training_artifacts"
CKPT_DIR        = ARTIFACTS_DIR / "checkpoints"
DATA_DIR        = ARTIFACTS_DIR / "data"
LOGS_DIR        = ARTIFACTS_DIR / "logs"
PAST_AGENTS_DIR = CKPT_DIR / "past_agents"
SUBMISSION_DIR  = REPO_DIR / "submissions" / "latest"

for d in [CKPT_DIR, DATA_DIR, LOGS_DIR, PAST_AGENTS_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Artifacts dir :", ARTIFACTS_DIR)
print("Submission dir:", SUBMISSION_DIR)


In [ ]:
import subprocess, sys, os

assert "REPO_DIR" in dir(), "REPO_DIR not defined — run Cell 1 first."

checks = [
    ("reward",   "from src.training.reward import compute_reward; print('reward OK')"),
    ("features", "from src.utils.feature_extractor import extract_features; print('features OK')"),
    ("network",  "from src.models.policy_network import BomberPolicyNet; print('network OK')"),
]

all_ok = True
_env = {**os.environ, "PYTHONPATH": str(REPO_DIR)}
for name, stmt in checks:
    result = subprocess.run(
        [sys.executable, "-c", stmt],
        capture_output=True, text=True,
        cwd=str(REPO_DIR),
        env=_env,
    )
    if result.returncode == 0:
        print(result.stdout.strip())
    else:
        print(f"FAIL ({name}):", result.stderr.strip())
        all_ok = False

assert all_ok, "One or more src/ imports failed — fix before running training."


## Phase 1: TacticalAgent Behavioral Cloning

In [ ]:
assert "REPO_DIR" in dir(), "REPO_DIR not defined — run Cell 1 first."

N_GAMES      = 200
DATA_DIR_STR = str(DATA_DIR)

%cd {REPO_DIR}
!python -m src.training.tactical_bc \
    --generate \
    --n-games {N_GAMES} \
    --data-dir {DATA_DIR_STR}

print(f"TacticalBC data generation complete: {DATA_DIR_STR}")


In [ ]:
assert "REPO_DIR" in dir(), "REPO_DIR not defined — run Cell 1 first."
assert "CKPT_DIR" in dir(), "CKPT_DIR not defined — run Cell 2 first."

BC_EPOCHS    = 20
DATA_DIR_STR = str(DATA_DIR)
CKPT_DIR_STR = str(CKPT_DIR)

%cd {REPO_DIR}
!python -m src.training.tactical_bc \
    --train \
    --epochs {BC_EPOCHS} \
    --data-dir {DATA_DIR_STR} \
    --output-dir {CKPT_DIR_STR}

print(f"TacticalBC training complete. Checkpoints in: {CKPT_DIR_STR}")


In [ ]:
assert "CKPT_DIR" in dir(), "CKPT_DIR not defined — run Cell 2 first."
from pathlib import Path

CHECKPOINT = ""

if not CHECKPOINT:
    candidates = sorted(
        CKPT_DIR.glob("tactical_bc_*.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert candidates, (
        f"No tactical_bc_*.pt found in {CKPT_DIR}. "
        "Run TacticalBC Cell B first, or set CHECKPOINT manually."
    )
    CHECKPOINT = str(candidates[0])

TACTICAL_BC_CKPT = CHECKPOINT
print(f"Using TacticalBC checkpoint: {TACTICAL_BC_CKPT}")


In [ ]:
assert "CKPT_DIR" in dir(), "CKPT_DIR not defined — run Cell 2 first."
from pathlib import Path

CHECKPOINT           = ""
PPO_STEPS_PER_STAGE  = 750_000
N_ENVS               = 4
EVAL_FREQ            = 50_000
EVAL_EPISODES        = 200
MIN_STEPS_PER_STAGE  = 100_000
DEVICE               = "auto"

CKPT_DIR_STR = str(CKPT_DIR)
LOGS_DIR_STR = str(LOGS_DIR)

if not CHECKPOINT:
    candidates = sorted(
        CKPT_DIR.glob("tactical_bc_*.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert candidates, (
        f"No tactical_bc_*.pt found in {CKPT_DIR}. "
        "Run TacticalBC Cell B and Cell 5 first, or set CHECKPOINT manually."
    )
    CHECKPOINT = str(candidates[0])

TACTICAL_BC_CKPT = CHECKPOINT
print(f"PPO init checkpoint:  {TACTICAL_BC_CKPT}")
print(f"Steps per stage:      {PPO_STEPS_PER_STAGE:,}")
print(f"Parallel envs:        {N_ENVS}")
print(f"VecNormalize stats:   {CKPT_DIR_STR}/vecnormalize_{{stage}}.pkl")

%cd {REPO_DIR}
!python -m src.training.ppo_trainer \
    --curriculum \
    --init-from-tactical {TACTICAL_BC_CKPT} \
    --output-dir {CKPT_DIR_STR} \
    --log-dir {LOGS_DIR_STR} \
    --total-steps-per-stage {PPO_STEPS_PER_STAGE} \
    --n-envs {N_ENVS} \
    --eval-freq {EVAL_FREQ} \
    --eval-episodes {EVAL_EPISODES} \
    --min-steps-per-stage {MIN_STEPS_PER_STAGE} \
    --device {DEVICE}

print("PPO curriculum complete.")


In [ ]:
assert "REPO_DIR" in dir(), "REPO_DIR not defined — run Cell 1 first."
assert "CKPT_DIR" in dir(), "CKPT_DIR not defined — run Cell 2 first."
from pathlib import Path

CHECKPOINT          = ""
SELFPLAY_STEPS      = 500_000
N_ENVS              = 4
SNAPSHOT_EVERY      = 50_000
POOL_SIZE           = 20
DEVICE              = "auto"

CKPT_DIR_STR        = str(CKPT_DIR)
PAST_AGENTS_DIR_STR = str(PAST_AGENTS_DIR)
LOGS_DIR_STR        = str(LOGS_DIR)

PAST_AGENTS_DIR.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT:
    candidates = sorted(
        list(CKPT_DIR.glob("ppo_*.pt")) + list(CKPT_DIR.glob("selfplay_*.pt")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        CHECKPOINT = str(candidates[0])
    else:
        print("WARNING: No PPO checkpoint found — skipping self-play.")
        CHECKPOINT = None

PPO_BEST_CKPT = CHECKPOINT

if PPO_BEST_CKPT:
    print(f"Self-play init checkpoint: {PPO_BEST_CKPT}")
    print(f"Total self-play steps:     {SELFPLAY_STEPS:,}")

    %cd {REPO_DIR}
    !python -m src.training.ppo_trainer \
        --self-play \
        --init-from {PPO_BEST_CKPT} \
        --output-dir {CKPT_DIR_STR} \
        --snapshot-dir {PAST_AGENTS_DIR_STR} \
        --log-dir {LOGS_DIR_STR} \
        --total-steps {SELFPLAY_STEPS} \
        --n-envs {N_ENVS} \
        --snapshot-every {SNAPSHOT_EVERY} \
        --pool-size {POOL_SIZE} \
        --device {DEVICE}

    print("Self-play complete.")
else:
    print("Self-play skipped — no PPO checkpoint available.")


In [ ]:
assert "CKPT_DIR" in dir(), "CKPT_DIR not defined — run Cell 2 first."
from pathlib import Path

CHECKPOINT  = ""
ONNX_OPSET  = 17
VERIFY_ONNX = True

ONNX_OUTPUT = str(CKPT_DIR / "model.onnx")

if not CHECKPOINT:
    for pattern in ("selfplay_*.pt", "ppo_*.pt", "tactical_bc_*.pt"):
        candidates = sorted(
            CKPT_DIR.glob(pattern),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            CHECKPOINT = str(candidates[0])
            break

assert CHECKPOINT, f"No checkpoint found in {CKPT_DIR}. Set CHECKPOINT manually."
BEST_CKPT = CHECKPOINT
NO_VERIFY_FLAG = "" if VERIFY_ONNX else "--no-verify"
print(f"Exporting checkpoint:  {BEST_CKPT}")
print(f"ONNX output:           {ONNX_OUTPUT}")
print(f"TorchScript output:    {str(CKPT_DIR / 'model.pt')}")

%cd {REPO_DIR}
!python -m src.utils.export_onnx \
    --checkpoint {BEST_CKPT} \
    --output {ONNX_OUTPUT} \
    --opset {ONNX_OPSET} \
    {NO_VERIFY_FLAG} \
    --torchscript

print("Export complete.")


In [ ]:
import shutil
import zipfile
import time
from pathlib import Path

assert "REPO_DIR" in dir(), "REPO_DIR not defined — run Cell 1 first."
assert "CKPT_DIR" in dir(), "CKPT_DIR not defined — run Cell 2 first."

SUBMISSION_DIR = REPO_DIR / "submissions" / "latest"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

AGENT_SRC = REPO_DIR / "agent" / "agent.py"
ONNX_SRC  = CKPT_DIR / "model.onnx"
PT_SRC    = CKPT_DIR / "model.pt"

AGENT_DST = SUBMISSION_DIR / "agent.py"
ONNX_DST  = SUBMISSION_DIR / "model.onnx"
PT_DST    = SUBMISSION_DIR / "model.pt"

for src, dst in [(AGENT_SRC, AGENT_DST), (ONNX_SRC, ONNX_DST), (PT_SRC, PT_DST)]:
    assert src.exists(), f"FATAL: source file missing: {src}"
    shutil.copy2(src, dst)
    print(f"Copied: {src.name} -> {dst}")

assert not (SUBMISSION_DIR / "requirements.txt").exists(), \
    "FATAL: requirements.txt is FORBIDDEN — remove it!"

ts = time.strftime("%Y%m%d_%H%M")
zip_path = REPO_DIR / "submissions" / f"submission_{ts}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [AGENT_DST, ONNX_DST, PT_DST]:
        zf.write(f, f.name)

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()

print("\nFiles in submission zip:")
for n in sorted(names):
    print(f"  {n}")

assert "agent.py" in names, "FATAL: agent.py not at root of submission.zip!"
print("\nagent.py at zip root:     OK")
print("model.onnx present:       OK")
print("model.pt present:         OK")
print("requirements.txt absent:  OK")
print(f"\nSubmission zip: {zip_path}")
